# Day 1 - Exercise 02: Building Your First LangChain Agent

## Real-World Scenario: Personal Research Assistant

**Context:** You're building a research assistant that can help users gather information from the web, perform calculations, and answer questions with cited sources.

**Learning Objectives:**
- Understand LangChain's core components
- Build an agent from scratch
- Integrate multiple tools
- Observe the ReAct (Reason + Act) pattern in action

---

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import create_agent

llm = ChatAnthropic(
    api_key=os.environ.get("KEY"),
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    base_url="https://llmgw-wp.tekstac.com"
)
print("Setup complete!")

---
## Part 1: Understanding the Agent Components

An agent has four core components:

| Component | Purpose | LangChain Class |
|-----------|---------|----------------|
| **LLM** | The "brain" that reasons | `ChatAnthropic` |
| **Tools** | Actions the agent can take | `@tool` decorated functions |
| **Prompt** | Instructions for the agent | `ChatPromptTemplate` |
| **Memory** | Conversation history | List of messages |

Let's build each one step by step.

### Step 1: Define Tools

Tools are Python functions that the agent can call. Each tool must have:
- A clear **name** (function name)
- A **description** (docstring) — the agent reads this to decide when to use the tool
- **Parameters** with type hints

In [ ]:
import math
from datetime import datetime

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Supports basic operations (+, -, *, /, **) and functions like sqrt, sin, cos.
    Examples: '2 + 2', '10 ** 2', 'sqrt(16)', '100 / 4'"""
    try:
        # Safe evaluation with limited functions
        allowed_names = {
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
            'tan': math.tan, 'log': math.log, 'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_datetime() -> str:
    """Get the current date and time. Use this when the user asks about today's date or current time."""
    now = datetime.now()
    return f"Current date and time: {now.strftime('%A, %B %d, %Y at %I:%M %p')}"

@tool
def search_knowledge_base(query: str) -> str:
    """Search our internal knowledge base for information about AI, Machine Learning, and LangChain."""
    # Simulated knowledge base
    knowledge = {
        "langchain": "LangChain is an open-source framework for building LLM applications. Created by Harrison Chase in 2022. Key components: Chains, Agents, Memory, Tools.",
        "agentic ai": "Agentic AI refers to AI systems that can plan, reason, and take autonomous actions to achieve goals. Key features: tool use, memory, multi-step reasoning.",
        "claude": "Claude is an AI assistant created by Anthropic. Known for being helpful, harmless, and honest. Supports up to 200K context window.",
        "react": "ReAct (Reason + Act) is a prompting pattern where the AI thinks step-by-step, takes actions, observes results, and repeats until done.",
        "rag": "RAG (Retrieval-Augmented Generation) combines LLMs with external knowledge retrieval. Steps: Query → Retrieve documents → Generate answer with context."
    }
    
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return f"Knowledge Base Result:\n{value}"
    return "No matching information found in the knowledge base."

# Collect all tools
tools = [calculator, get_current_datetime, search_knowledge_base]
print("Tools defined:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

### Step 2: Create the Prompt Template

The prompt tells the agent:
- Who it is (system message)
- What it can do (tools are automatically injected)
- How to behave

In [ ]:
system_prompt = """You are a helpful research assistant with access to tools.

Your capabilities:
- Calculate mathematical expressions
- Get the current date and time
- Search our knowledge base about AI topics

Guidelines:
1. Always use tools when you need factual information
2. Think step-by-step before answering complex questions
3. Cite your sources when using the knowledge base
4. If you don't know something, say so honestly
"""

print("System prompt created!")

### Step 3: Assemble the Agent

Now we combine the LLM, tools, and prompt into an agent.

In [ ]:
# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agent created!")

### Step 4: Add Memory

Memory is simply a list of messages that gets passed to the agent each time.

In [ ]:
chat_history = []  # Our memory store

def chat(user_input: str) -> str:
    """Send a message to the agent and update memory."""
    global chat_history
    
    # Build messages list: chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Chat function ready!")

---
## Part 2: Testing the Agent

Let's see the agent in action with different types of questions.

In [ ]:
# Test 1: Calculator tool
print("="*60)
print("Test 1: Math calculation")
print("="*60)
response = chat("What is 15% of 250?")
print(f"\nFinal Answer: {response}")

In [ ]:
# Test 2: Knowledge base search
print("="*60)
print("Test 2: Knowledge lookup")
print("="*60)
response = chat("What is LangChain and who created it?")
print(f"\nFinal Answer: {response}")

In [ ]:
# Test 3: Multi-tool usage
print("="*60)
print("Test 3: Complex question requiring multiple tools")
print("="*60)
response = chat("What is today's date, and if LangChain was created in 2022, how many years ago was that?")
print(f"\nFinal Answer: {response}")

In [ ]:
# Test 4: Memory test
print("="*60)
print("Test 4: Memory - follow-up question")
print("="*60)
response = chat("Based on what you told me earlier, what are the key components of LangChain?")
print(f"\nFinal Answer: {response}")

---
## Part 3: Observing the ReAct Pattern

The `verbose=True` setting shows us the agent's thinking process:

```
Thought: I need to calculate 15% of 250
Action: calculator
Action Input: "250 * 0.15"
Observation: Result: 37.5
Thought: I now have the answer
Final Answer: 15% of 250 is 37.5
```

This is the **ReAct loop**: Reason → Act → Observe → Repeat

---
##  Your Turn: Practice Exercises

### Exercise 2.1: Add a New Tool

Create a tool that converts between units (temperature, length, weight).

In [ ]:
# TODO: Create a unit converter tool

@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between units. Supports:
    - Temperature: celsius, fahrenheit, kelvin
    - Length: meters, feet, inches, kilometers, miles
    - Weight: kg, pounds, ounces
    """
    # YOUR CODE HERE
    # Example conversions:
    # celsius to fahrenheit: (value * 9/5) + 32
    # meters to feet: value * 3.28084
    # kg to pounds: value * 2.20462
    pass

# Test your tool
# result = convert_units.invoke({"value": 100, "from_unit": "celsius", "to_unit": "fahrenheit"})
# print(result)

### Exercise 2.2: Customize the Agent Persona

Modify the system prompt to create a **Financial Advisor** agent that:
- Uses formal language
- Always provides disclaimers
- Focuses on financial calculations and advice

In [ ]:
# TODO: Create a financial advisor prompt

'''EXample:
 
You are a professional financial advisor assistant.
    
# YOUR INSTRUCTIONS HERE:
# - Define the persona
# - Set guidelines for responses
# - Add disclaimer requirements


# Create and test the financial advisor agent
'''

### Exercise 2.3: Build a Complete Agent

Build a **Travel Planning Agent** with these tools:
1. `get_weather(city)` — Returns weather for a city
2. `search_flights(from_city, to_city, date)` — Searches for flights
3. `get_attractions(city)` — Lists popular attractions

Use simulated data for the tools.

In [ ]:
# TODO: Build a Travel Planning Agent

# Step 1: Define simulated data
WEATHER_DATA = {
    "paris": "Sunny, 22°C",
    "tokyo": "Cloudy, 18°C",
    "new york": "Rainy, 15°C"
}

ATTRACTIONS = {
    "paris": ["Eiffel Tower", "Louvre Museum", "Notre-Dame"],
    "tokyo": ["Tokyo Tower", "Senso-ji Temple", "Shibuya Crossing"],
    "new york": ["Statue of Liberty", "Central Park", "Times Square"]
}

# Step 2: Define tools
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    # YOUR CODE HERE
    pass

@tool
def search_flights(from_city: str, to_city: str, date: str) -> str:
    """Search for available flights."""
    # YOUR CODE HERE
    pass

@tool
def get_attractions(city: str) -> str:
    """Get popular tourist attractions in a city."""
    # YOUR CODE HERE
    pass

# Step 3: Create the agent
# YOUR CODE HERE

# Step 4: Test with: "I'm planning a trip to Paris next week. What's the weather like and what should I see?"

---
##  Key Takeaways

1. **LangChain agents** combine LLM + Tools + Prompt + Memory
2. **Tools** are Python functions with clear descriptions — the agent reads docstrings to decide when to use them
3. **The prompt** defines the agent's persona and behavior
4. **Memory** is a list of messages passed to each agent call
5. **AgentExecutor** runs the ReAct loop automatically
6. **verbose=True** shows the agent's reasoning — essential for debugging

---

**Next Exercise →** Agent Workflows and Real-World Use Cases